# 02. Data Preprocessing – Pima Diabetes Dataset

## Master’s Dissertation Pipeline

This notebook prepares the Pima Diabetes dataset for baseline modelling, explainability analysis, counterfactual recourse, and the EES framework.

The preprocessing design follows the same standardised approach used for the German Credit dataset, while incorporating a dataset-specific clinical treatment for physiologically implausible zero values.

## Main Objectives

1. Load the raw Pima Diabetes dataset.
2. Treat clinically implausible zero values as missing values.
3. Split the data into training and test sets.
4. Fit preprocessing transformations on the training set only.
5. Apply transformations consistently to training and test data.
6. Save reproducible modelling inputs for downstream notebooks.
7. Create audit-ready preprocessing documentation.

# 1. Dissertation Alignment

This notebook contributes to the reproducible machine learning pipeline required for the Explainability–Ethics Scoring (EES) framework.

The preprocessing approach is intentionally standardised across datasets to support cross-domain benchmarking. Dataset-specific treatment is applied only where defensible, such as the clinical interpretation of zero values in the Pima Diabetes dataset.

## Dataset-Specific Consideration

In the Pima Diabetes dataset, several clinical variables contain zero values that are physiologically implausible. These are interpreted as missing observations rather than true measurements.

# 2. Key Preprocessing Assumptions

| Area | Assumption | Defensible Reason |
|---|---|---|
| Target | `Outcome` is the binary target variable | Standard Pima Diabetes target, where 1 indicates diabetes and 0 indicates non-diabetes |
| Clinically implausible zeros | Zeros in selected clinical variables are treated as missing | Zero values for glucose, blood pressure, skin thickness, insulin, and BMI are not clinically meaningful |
| Imputation | Median imputation is used | Median is robust to skewness and outliers in clinical measurements |
| Scaling | RobustScaler is used | RobustScaler reduces sensitivity to outliers while preserving distributional structure |
| Data leakage | Preprocessor is fitted on training data only | Prevents information from the test set leaking into model development |
| Train/test split | Stratified 80/20 split | Preserves class balance across train and test sets |
| Reproducibility | `random_state = 42` | Ensures repeatability across runs |
| Cross-domain consistency | Same output structure as German Credit | Enables replication across diabetes and recidivism datasets |

In [1]:
# =========================================================
# 3. Import Libraries
# =========================================================

from pathlib import Path
import warnings
import joblib

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
# =========================================================
# 4. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")
DATASET_NAME = "pima_diabetes"
NOTEBOOK_NAME = "02_data_preprocessing_pima_diabetes"

RAW_DATA_PATH = ROOT_PATH / "04_Data" / "01_Raw_Data" / DATASET_NAME / "diabetes.csv"
RESULTS_PATH = ROOT_PATH / "07_Results" / DATASET_NAME / NOTEBOOK_NAME

DATASETS_PATH = RESULTS_PATH / "datasets"
TABLES_PATH = RESULTS_PATH / "tables"
MODELS_PATH = RESULTS_PATH / "models"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, DATASETS_PATH, TABLES_PATH, MODELS_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Project setup complete.")
print(f"Root path:       {ROOT_PATH}")
print(f"Raw data path:   {RAW_DATA_PATH}")
print(f"Results path:    {RESULTS_PATH}")

Mounted at /content/drive
Project setup complete.
Root path:       /content/drive/MyDrive/Master_Research
Raw data path:   /content/drive/MyDrive/Master_Research/04_Data/01_Raw_Data/pima_diabetes/diabetes.csv
Results path:    /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes


In [3]:
# =========================================================
# 5. Helper Functions
# =========================================================

def save_df(df, filename, folder=TABLES_PATH):
    """Save dataframe as CSV and return output path."""
    output_path = folder / filename
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")
    return output_path


def validate_file_exists(path, label):
    """Raise a clear error if a required file is missing."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found at: {path}")
    return path

In [4]:
# =========================================================
# 6. Load Raw Pima Diabetes Dataset
# =========================================================

validate_file_exists(RAW_DATA_PATH, "Raw Pima Diabetes dataset")

diabetes_df = pd.read_csv(RAW_DATA_PATH)

print("Pima Diabetes dataset loaded successfully.")
print(f"Shape: {diabetes_df.shape}")
display(diabetes_df.head())

Pima Diabetes dataset loaded successfully.
Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
# =========================================================
# 7. Basic Dataset Validation
# =========================================================

expected_columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

missing_expected_cols = [col for col in expected_columns if col not in diabetes_df.columns]

if missing_expected_cols:
    raise ValueError(
        f"The following expected columns are missing: {missing_expected_cols}. "
        f"Available columns: {diabetes_df.columns.tolist()}"
    )

# Keep only the expected columns in the standard order
diabetes_df = diabetes_df[expected_columns].copy()

validation_checks = []
validation_checks.append({"check": "Dataset has rows", "passed": len(diabetes_df) > 0})
validation_checks.append({"check": "Outcome column exists", "passed": "Outcome" in diabetes_df.columns})
validation_checks.append({"check": "Outcome is binary", "passed": sorted(diabetes_df["Outcome"].dropna().unique().tolist()) == [0, 1]})
validation_checks.append({"check": "No duplicate rows", "passed": diabetes_df.duplicated().sum() == 0})

validation_df = pd.DataFrame(validation_checks)
display(validation_df)

if not validation_df["passed"].all():
    raise ValueError("One or more basic dataset validation checks failed.")

save_df(validation_df, "basic_dataset_validation.csv")

,check,passed
0,Dataset has rows,True
1,Outcome column exists,True
2,Outcome is binary,True
3,No duplicate rows,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/basic_dataset_validation.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/basic_dataset_validation.csv')

# 8. Clinically Implausible Zero Treatment

The Pima Diabetes dataset contains zeros in variables where zero is not clinically plausible. These values are treated as missing values.

The affected variables are:

- `Glucose`
- `BloodPressure`
- `SkinThickness`
- `Insulin`
- `BMI`

This treatment is applied before train/test splitting so that the split mapping reflects the cleaned analytical dataset. However, imputation is fitted only on the training data to avoid data leakage.

In [6]:
# =========================================================
# 8. Zero-to-Missing Summary
# =========================================================

ZERO_AS_MISSING_FEATURES = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

zero_summary_rows = []

for feature in ZERO_AS_MISSING_FEATURES:
    n_zero = int((diabetes_df[feature] == 0).sum())
    zero_pct = (n_zero / len(diabetes_df)) * 100
    zero_summary_rows.append({
        "feature": feature,
        "n_zero_values": n_zero,
        "zero_value_pct": zero_pct,
        "preprocessing_action": "Convert 0 to NaN",
        "imputation_method": "Median imputation fitted on training data only",
        "rationale": "Zero is clinically implausible and is treated as a missing observation."
    })

zero_to_nan_summary_df = pd.DataFrame(zero_summary_rows)
display(zero_to_nan_summary_df)
save_df(zero_to_nan_summary_df, "zero_to_nan_summary.csv")

,feature,n_zero_values,zero_value_pct,preprocessing_action,imputation_method,rationale
0,Glucose,5,0.651042,Convert 0 to NaN,Median imputation fitted on training data only,Zero is clinically implausible and is treated ...
1,BloodPressure,35,4.557292,Convert 0 to NaN,Median imputation fitted on training data only,Zero is clinically implausible and is treated ...
2,SkinThickness,227,29.557292,Convert 0 to NaN,Median imputation fitted on training data only,Zero is clinically implausible and is treated ...
3,Insulin,374,48.697917,Convert 0 to NaN,Median imputation fitted on training data only,Zero is clinically implausible and is treated ...
4,BMI,11,1.432292,Convert 0 to NaN,Median imputation fitted on training data only,Zero is clinically implausible and is treated ...


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/zero_to_nan_summary.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/zero_to_nan_summary.csv')

In [7]:
# =========================================================
# 9. Apply Zero-to-NaN Conversion
# =========================================================

diabetes_clean_df = diabetes_df.copy()

diabetes_clean_df[ZERO_AS_MISSING_FEATURES] = (
    diabetes_clean_df[ZERO_AS_MISSING_FEATURES]
    .replace(0, np.nan)
)

missing_after_zero_conversion_df = pd.DataFrame({
    "feature": diabetes_clean_df.columns,
    "missing_count": diabetes_clean_df.isna().sum().values,
    "missing_pct": (diabetes_clean_df.isna().mean().values * 100)
})

missing_after_zero_conversion_df = missing_after_zero_conversion_df.sort_values(
    "missing_pct", ascending=False
).reset_index(drop=True)

display(missing_after_zero_conversion_df)
save_df(missing_after_zero_conversion_df, "missing_after_zero_conversion.csv")

,feature,missing_count,missing_pct
0,Insulin,374,48.697917
1,SkinThickness,227,29.557292
2,BloodPressure,35,4.557292
3,BMI,11,1.432292
4,Glucose,5,0.651042
5,Pregnancies,0,0.000000
6,DiabetesPedigreeFunction,0,0.000000
7,Age,0,0.000000
8,Outcome,0,0.000000


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/missing_after_zero_conversion.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/missing_after_zero_conversion.csv')

In [8]:
# =========================================================
# 10. Feature Audit Table
# =========================================================

feature_metadata = {
    "Pregnancies": {
        "description": "Number of pregnancies",
        "sensitive": False,
        "mutable": False,
        "notes": "Historical reproductive variable; not generally actionable."
    },
    "Glucose": {
        "description": "Plasma glucose concentration",
        "sensitive": False,
        "mutable": True,
        "notes": "Potentially modifiable through clinical and lifestyle interventions."
    },
    "BloodPressure": {
        "description": "Diastolic blood pressure",
        "sensitive": False,
        "mutable": True,
        "notes": "Potentially modifiable with treatment and lifestyle changes."
    },
    "SkinThickness": {
        "description": "Triceps skinfold thickness",
        "sensitive": False,
        "mutable": True,
        "notes": "Body composition indicator; may be modifiable over time."
    },
    "Insulin": {
        "description": "Two-hour serum insulin",
        "sensitive": False,
        "mutable": True,
        "notes": "Clinical marker; may change with treatment."
    },
    "BMI": {
        "description": "Body mass index",
        "sensitive": False,
        "mutable": True,
        "notes": "Potentially modifiable over time; important for recourse interpretation."
    },
    "DiabetesPedigreeFunction": {
        "description": "Diabetes pedigree function",
        "sensitive": False,
        "mutable": False,
        "notes": "Family-history proxy; not directly actionable."
    },
    "Age": {
        "description": "Age in years",
        "sensitive": True,
        "mutable": False,
        "notes": "Sensitive demographic variable; not actionable."
    }
}

feature_audit_rows = []

for feature in [col for col in diabetes_clean_df.columns if col != "Outcome"]:
    series = diabetes_clean_df[feature]
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = int(((series < lower) | (series > upper)).sum(skipna=True))
    outlier_pct = outlier_count / len(series) * 100

    meta = feature_metadata.get(feature, {})

    feature_audit_rows.append({
        "feature": feature,
        "feature_description": meta.get("description", "Not specified"),
        "data_type": "numeric",
        "sensitive": meta.get("sensitive", False),
        "mutable": meta.get("mutable", True),
        "missing_count": int(series.isna().sum()),
        "missing_pct": series.isna().mean() * 100,
        "unique_values": int(series.nunique(dropna=True)),
        "min": series.min(),
        "median": series.median(),
        "max": series.max(),
        "skewness": series.skew(),
        "outlier_count_iqr": outlier_count,
        "outlier_pct_iqr": outlier_pct,
        "zero_converted_to_missing": feature in ZERO_AS_MISSING_FEATURES,
        "imputation_method": "Median" if feature in ZERO_AS_MISSING_FEATURES else "Median if missing",
        "scaling_applied": "RobustScaler",
        "encoding_applied": "None - numeric feature",
        "proxy_risk": "Possible" if feature in ["Age", "DiabetesPedigreeFunction"] else "Low",
        "notes": meta.get("notes", "")
    })

feature_audit_df = pd.DataFrame(feature_audit_rows)
display(feature_audit_df)
save_df(feature_audit_df, "feature_audit_table.csv")

,feature,feature_description,data_type,sensitive,mutable,missing_count,missing_pct,unique_values,min,median,max,skewness,outlier_count_iqr,outlier_pct_iqr,zero_converted_to_missing,imputation_method,scaling_applied,encoding_applied,proxy_risk,notes
0,Pregnancies,Number of pregnancies,numeric,False,False,0,0.000000,17,0.000,3.0000,17.00,0.901674,4,0.520833,False,Median if missing,RobustScaler,None - numeric feature,Low,Historical reproductive variable; not generall...
1,Glucose,Plasma glucose concentration,numeric,False,True,5,0.651042,135,44.000,117.0000,199.00,0.530989,0,0.000000,True,Median,RobustScaler,None - numeric feature,Low,Potentially modifiable through clinical and li...
2,BloodPressure,Diastolic blood pressure,numeric,False,True,35,4.557292,46,24.000,72.0000,122.00,0.134153,14,1.822917,True,Median,RobustScaler,None - numeric feature,Low,Potentially modifiable with treatment and life...
3,SkinThickness,Triceps skinfold thickness,numeric,False,True,227,29.557292,50,7.000,29.0000,99.00,0.690619,3,0.390625,True,Median,RobustScaler,None - numeric feature,Low,Body composition indicator; may be modifiable ...
4,Insulin,Two-hour serum insulin,numeric,False,True,374,48.697917,185,14.000,125.0000,846.00,2.166464,24,3.125000,True,Median,RobustScaler,None - numeric feature,Low,Clinical marker; may change with treatment.
5,BMI,Body mass index,numeric,False,True,11,1.432292,247,18.200,32.3000,67.10,0.593970,8,1.041667,True,Median,RobustScaler,None - numeric feature,Low,Potentially modifiable over time; important fo...
6,DiabetesPedigreeFunction,Diabetes pedigree function,numeric,False,False,0,0.000000,517,0.078,0.3725,2.42,1.919911,29,3.776042,False,Median if missing,RobustScaler,None - numeric feature,Possible,Family-history proxy; not directly actionable.
7,Age,Age in years,numeric,True,False,0,0.000000,52,21.000,29.0000,81.00,1.129597,9,1.171875,False,Median if missing,RobustScaler,None - numeric feature,Possible,Sensitive demographic variable; not actionable.


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/feature_audit_table.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/feature_audit_table.csv')

In [9]:
# =========================================================
# 11. Separate Features and Target
# =========================================================

TARGET_COLUMN = "Outcome"

X = diabetes_clean_df.drop(columns=[TARGET_COLUMN]).copy()
y = diabetes_clean_df[TARGET_COLUMN].astype(int).copy()

# Store original row index for traceability before splitting
X = X.reset_index().rename(columns={"index": "original_index"})
y_df = pd.DataFrame({
    "original_index": X["original_index"],
    "target": y.reset_index(drop=True)
})

print("Feature shape:", X.shape)
print("Target shape:", y_df.shape)
display(X.head())
display(y_df.head())

Feature shape: (768, 9)
Target shape: (768, 2)


,original_index,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,0,6,148.0,72.0,35.0,NaN,33.6,0.627,50
1,1,1,85.0,66.0,29.0,NaN,26.6,0.351,31
2,2,8,183.0,64.0,NaN,NaN,23.3,0.672,32
3,3,1,89.0,66.0,23.0,94.0,28.1,0.167,21
4,4,0,137.0,40.0,35.0,168.0,43.1,2.288,33


,original_index,target
0,0,1
1,1,0
2,2,1
3,3,0
4,4,1


In [10]:
# =========================================================
# 12. Train/Test Split
# =========================================================

RANDOM_STATE = 42
TEST_SIZE = 0.20

X_train_original, X_test_original, y_train_df, y_test_df = train_test_split(
    X,
    y_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_df["target"]
)

# Reset row positions while preserving original_index column
X_train_original = X_train_original.reset_index(drop=True)
X_test_original = X_test_original.reset_index(drop=True)
y_train_df = y_train_df.reset_index(drop=True)
y_test_df = y_test_df.reset_index(drop=True)

y_train = y_train_df["target"].astype(int)
y_test = y_test_df["target"].astype(int)

split_index_mapping_df = pd.concat([
    pd.DataFrame({
        "split": "train",
        "split_position": np.arange(len(X_train_original)),
        "original_index": X_train_original["original_index"].values,
        "target": y_train.values
    }),
    pd.DataFrame({
        "split": "test",
        "split_position": np.arange(len(X_test_original)),
        "original_index": X_test_original["original_index"].values,
        "target": y_test.values
    })
], ignore_index=True)

print("Train shape:", X_train_original.shape)
print("Test shape:", X_test_original.shape)
print("Train target distribution:")
display(y_train.value_counts(normalize=True).rename("proportion").reset_index())
print("Test target distribution:")
display(y_test.value_counts(normalize=True).rename("proportion").reset_index())

save_df(split_index_mapping_df, "split_index_mapping.csv")

Train shape: (614, 9)
Test shape: (154, 9)
Train target distribution:


,target,proportion
0,0,0.651466
1,1,0.348534


Test target distribution:


,target,proportion
0,0,0.649351
1,1,0.350649


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/split_index_mapping.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/split_index_mapping.csv')

# 13. Preprocessing Design

The Pima Diabetes dataset contains numeric variables only. Therefore, the preprocessing pipeline consists of:

1. median imputation for missing values;
2. robust scaling for numeric variables.

The tracking column `original_index` is preserved in the original split files but excluded from model features and preprocessing.

In [11]:
# =========================================================
# 13. Define Preprocessing Features
# =========================================================

TRACKING_COLUMNS = ["original_index"]

X_train_model_original = X_train_original.drop(columns=TRACKING_COLUMNS, errors="ignore").copy()
X_test_model_original = X_test_original.drop(columns=TRACKING_COLUMNS, errors="ignore").copy()

numeric_features = X_train_model_original.columns.tolist()
categorical_features = []

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
Categorical features: []


In [13]:
# =========================================================
# 14. Build and Fit Preprocessor
# =========================================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features)
    ],
    remainder="drop"
)

preprocessor.fit(X_train_model_original)

print("Preprocessor fitted successfully on training data only.")

Preprocessor fitted successfully on training data only.


In [14]:
# =========================================================
# 15. Transform Train and Test Data
# =========================================================

X_train_processed_array = preprocessor.transform(X_train_model_original)
X_test_processed_array = preprocessor.transform(X_test_model_original)

preprocessed_feature_names = preprocessor.get_feature_names_out()

# Clean feature names for downstream readability
preprocessed_feature_names_clean = [
    name.replace("numeric__", "")
    for name in preprocessed_feature_names
]

X_train_preprocessed = pd.DataFrame(
    X_train_processed_array,
    columns=preprocessed_feature_names_clean
)

X_test_preprocessed = pd.DataFrame(
    X_test_processed_array,
    columns=preprocessed_feature_names_clean
)

print("X_train_preprocessed shape:", X_train_preprocessed.shape)
print("X_test_preprocessed shape:", X_test_preprocessed.shape)
display(X_train_preprocessed.head())

X_train_preprocessed shape: (614, 8)
X_test_preprocessed shape: (154, 8)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,-0.4,-0.658537,-0.625,-2.428571,-8.2,-0.585915,0.500951,-0.294118
1,0.4,0.219512,0.375,-0.285714,-10.3,-0.315493,0.143310,0.647059
2,-0.2,-0.292683,-0.875,1.571429,-3.1,0.281690,-0.399493,-0.235294
3,-0.4,0.707317,-1.000,0.000000,0.0,-0.304225,0.460368,0.000000
4,-0.6,-0.536585,-0.500,1.428571,-2.0,1.374648,-0.041852,-0.411765


In [15]:
# =========================================================
# 16. Validate Preprocessing Outputs
# =========================================================

validation_checks = []
validation_checks.append({"check": "X_train and y_train row alignment", "passed": len(X_train_preprocessed) == len(y_train)})
validation_checks.append({"check": "X_test and y_test row alignment", "passed": len(X_test_preprocessed) == len(y_test)})
validation_checks.append({"check": "No missing values in X_train_preprocessed", "passed": X_train_preprocessed.isna().sum().sum() == 0})
validation_checks.append({"check": "No missing values in X_test_preprocessed", "passed": X_test_preprocessed.isna().sum().sum() == 0})
validation_checks.append({"check": "Train and test have same preprocessed columns", "passed": list(X_train_preprocessed.columns) == list(X_test_preprocessed.columns)})
validation_checks.append({"check": "Binary target", "passed": sorted(y_train.unique().tolist()) == [0, 1]})
validation_checks.append({"check": "original_index retained in original feature splits", "passed": "original_index" in X_train_original.columns and "original_index" in X_test_original.columns})
validation_checks.append({"check": "original_index excluded from preprocessed features", "passed": "original_index" not in X_train_preprocessed.columns})

preprocessing_validation_df = pd.DataFrame(validation_checks)
display(preprocessing_validation_df)

if not preprocessing_validation_df["passed"].all():
    raise ValueError("One or more preprocessing validation checks failed.")

save_df(preprocessing_validation_df, "preprocessing_validation_checks.csv")

,check,passed
0,X_train and y_train row alignment,True
1,X_test and y_test row alignment,True
2,No missing values in X_train_preprocessed,True
3,No missing values in X_test_preprocessed,True
4,Train and test have same preprocessed columns,True
5,Binary target,True
6,original_index retained in original feature sp...,True
7,original_index excluded from preprocessed feat...,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/preprocessing_validation_checks.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/preprocessing_validation_checks.csv')

In [16]:
# =========================================================
# 17. Save Preprocessed Datasets and Artefacts
# =========================================================

# Original splits retain original_index for traceability
X_train_original.to_csv(DATASETS_PATH / "X_train_original.csv", index=False)
X_test_original.to_csv(DATASETS_PATH / "X_test_original.csv", index=False)

# Preprocessed model-ready datasets exclude original_index
X_train_preprocessed.to_csv(DATASETS_PATH / "X_train_preprocessed.csv", index=False)
X_test_preprocessed.to_csv(DATASETS_PATH / "X_test_preprocessed.csv", index=False)

# Targets retain original_index and target
# This mirrors the German Credit standard and avoids losing traceability
y_train_df.to_csv(DATASETS_PATH / "y_train.csv", index=False)
y_test_df.to_csv(DATASETS_PATH / "y_test.csv", index=False)

# Save feature names and preprocessing metadata
preprocessed_feature_names_df = pd.DataFrame({
    "feature": preprocessed_feature_names_clean
})
preprocessed_feature_names_df.to_csv(DATASETS_PATH / "preprocessed_feature_names.csv", index=False)

preprocessing_metadata_df = pd.DataFrame([
    {"item": "dataset_name", "value": DATASET_NAME},
    {"item": "target_column", "value": TARGET_COLUMN},
    {"item": "test_size", "value": TEST_SIZE},
    {"item": "random_state", "value": RANDOM_STATE},
    {"item": "zero_as_missing_features", "value": ", ".join(ZERO_AS_MISSING_FEATURES)},
    {"item": "numeric_imputation", "value": "median"},
    {"item": "scaling", "value": "RobustScaler"},
    {"item": "categorical_encoding", "value": "None - numeric dataset"},
    {"item": "tracking_columns", "value": ", ".join(TRACKING_COLUMNS)}
])
preprocessing_metadata_df.to_csv(TABLES_PATH / "preprocessing_metadata.csv", index=False)

# Save fitted preprocessor for future full pipeline construction
joblib.dump(preprocessor, MODELS_PATH / "preprocessor.pkl")

print("Saved preprocessing outputs successfully.")
print(f"Datasets path: {DATASETS_PATH}")
print(f"Tables path:   {TABLES_PATH}")
print(f"Models path:   {MODELS_PATH}")

Saved preprocessing outputs successfully.
Datasets path: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/datasets
Tables path:   /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables
Models path:   /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/models


In [17]:
# =========================================================
# 18. Create Preprocessing Output Inventory
# =========================================================

output_inventory = []

for folder_label, folder_path in {
    "datasets": DATASETS_PATH,
    "tables": TABLES_PATH,
    "models": MODELS_PATH,
    "logs": LOGS_PATH
}.items():
    for file in sorted(folder_path.glob("*")):
        if file.is_file():
            output_inventory.append({
                "folder": folder_label,
                "file_name": file.name,
                "file_path": str(file),
                "file_size_kb": round(file.stat().st_size / 1024, 2)
            })

output_inventory_df = pd.DataFrame(output_inventory)
display(output_inventory_df)
save_df(output_inventory_df, "preprocessing_output_inventory.csv")

,folder,file_name,file_path,file_size_kb
0,datasets,X_test_original.csv,/content/drive/MyDrive/Master_Research/07_Resu...,5.67
1,datasets,X_test_preprocessed.csv,/content/drive/MyDrive/Master_Research/07_Resu...,15.91
2,datasets,X_train_original.csv,/content/drive/MyDrive/Master_Research/07_Resu...,22.67
3,datasets,X_train_preprocessed.csv,/content/drive/MyDrive/Master_Research/07_Resu...,63.44
4,datasets,preprocessed_feature_names.csv,/content/drive/MyDrive/Master_Research/07_Resu...,0.09
5,datasets,y_test.csv,/content/drive/MyDrive/Master_Research/07_Resu...,0.90
6,datasets,y_train.csv,/content/drive/MyDrive/Master_Research/07_Resu...,3.53
7,tables,basic_dataset_validation.csv,/content/drive/MyDrive/Master_Research/07_Resu...,0.11
8,tables,feature_audit_table.csv,/content/drive/MyDrive/Master_Research/07_Resu...,2.10
9,tables,missing_after_zero_conversion.csv,/content/drive/MyDrive/Master_Research/07_Resu...,0.26


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/preprocessing_output_inventory.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/02_data_preprocessing_pima_diabetes/tables/preprocessing_output_inventory.csv')

# 19. Interpretation Note

The preprocessing pipeline successfully prepares the Pima Diabetes dataset for the standardised research workflow.

## Simple Interpretation

The dataset was cleaned by treating clinically impossible zero values as missing, then filling missing values using the median from the training data. The final model-ready datasets contain no missing values and can be used consistently across all candidate models.

## Technical Interpretation

The preprocessing pipeline applies median imputation and robust scaling to numeric clinical variables. The preprocessor is fitted only on the training split and then applied to both train and test splits, preventing data leakage. Original row identifiers are preserved for traceability but excluded from the model-ready feature matrix.

## Dissertation Contribution

This notebook creates a reproducible preprocessing layer that aligns with the German Credit pipeline while incorporating Pima-specific clinical assumptions. It supports cross-domain benchmarking by maintaining a standard output structure across datasets.

# 20. Downstream Outputs Required by Later Notebooks

The following outputs are required by downstream notebooks:

| Output | Used By |
|---|---|
| `X_train_original.csv` | Baseline modelling, DiCE, traceability |
| `X_test_original.csv` | Baseline modelling, DiCE, traceability |
| `X_train_preprocessed.csv` | Baseline modelling, SHAP, LIME |
| `X_test_preprocessed.csv` | Baseline modelling, SHAP, LIME |
| `y_train.csv` | Modelling and evaluation |
| `y_test.csv` | Modelling and evaluation |
| `split_index_mapping.csv` | Instance-level traceability |
| `feature_audit_table.csv` | Recourse, fairness/proxy-risk awareness |
| `zero_to_nan_summary.csv` | Missing-data documentation |
| `preprocessor.pkl` | Full pipeline construction in baseline modelling |